In [2]:
pip install -U sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [3]:
from sentence_transformers import SentenceTransformer, util
import torch

# 1. 加载本地模型 (第一次运行会自动下载到本地缓存)
model = SentenceTransformer('BAAI/bge-small-zh-v1.5')

# 2. 模拟你的《红楼梦》数据库 (建议按段落切分)
corpus = [
    "林黛玉进贾府，步步留心，时时在意，不肯多说一句话，多行一步路。",
    "宝玉看了，又道：'这个妹妹我曾见过的。' 贾母笑道：'可又是胡说！'",
    "黛玉方进入房中，见两个人搀着一个鬓发如银的老母，便知是贾母了。",
    "刘姥姥二进大观园，闹了许多笑话。"
]

# 3. 将全书片段转为向量 (这就是 Embedding 过程)
corpus_embeddings = model.encode(corpus, convert_to_tensor=True)

# 4. 定义你的测试问题 (Test1)
query = "宝玉第一次见黛玉是什么情景？"
query_embedding = model.encode(query, convert_to_tensor=True)

# 5. 计算余弦相似度并排序
cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
top_results = torch.topk(cos_scores, k=2)

print(f"\n查询问题: {query}")
for score, idx in zip(top_results[0], top_results[1]):
    print(f"得分: {score:.4f} | 匹配段落: {corpus[idx]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


查询问题: 宝玉第一次见黛玉是什么情景？
得分: 0.6420 | 匹配段落: 宝玉看了，又道：'这个妹妹我曾见过的。' 贾母笑道：'可又是胡说！'
得分: 0.5804 | 匹配段落: 黛玉方进入房中，见两个人搀着一个鬓发如银的老母，便知是贾母了。


In [5]:
# 使用已生成的 embeddings 和 chunks 进行查询测试
from sentence_transformers import SentenceTransformer, util
import torch
import pickle

# 1. 加载模型
print("加载模型...")
model = SentenceTransformer('BAAI/bge-small-zh-v1.5')

# 2. 加载已生成的 embeddings 和 chunks
print("加载 embeddings 和 chunks...")
corpus_embeddings = torch.load('corpus_embeddings.pt')
with open('corpus_chunks.pkl', 'rb') as f:
    corpus = pickle.load(f)

print(f"已加载 {len(corpus)} 个chunks")
print(f"Embeddings 形状: {corpus_embeddings.shape}")

# 3. 测试查询
query = "蘅芜院是谁的住所？"
print(f"\n查询: {query}")

# 4. 生成查询的 embedding
query_embedding = model.encode(query, convert_to_tensor=True, normalize_embeddings=True)

# 5. 计算余弦相似度并排序
cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
top_results = torch.topk(cos_scores, k=5)

# 6. 显示结果
print("\n" + "="*80)
print("Top 5 最相似的段落:")
print("="*80)
for i, (score, idx) in enumerate(zip(top_results[0], top_results[1]), 1):
    print(f"\n【排名 {i}】相似度: {score:.4f}")
    print(f"Chunk {idx+1} (长度: {len(corpus[idx])} 字符)")
    print(f"内容: {corpus[idx]}")
    print("-"*80)

加载模型...


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


加载 embeddings 和 chunks...
已加载 2636 个chunks
Embeddings 形状: torch.Size([2636, 512])

查询: 蘅芜院是谁的住所？

Top 5 最相似的段落:

【排名 1】相似度: 0.4413
Chunk 10 (长度: 209 字符)
内容: 按那石上书云：当日地陷东南，这东南有个姑苏城，城中阊门，最是红尘中一二等富贵风流之地。这阊门外有个十里街，街内有个仁清巷，巷内有个古庙，因地方狭窄，人皆呼作“葫芦庙”。庙旁住着一家乡宦，姓甄名费字士隐，嫡妻封氏，性情贤淑，深明礼义。家中虽不甚富贵，然本地也推他为望族了。因这甄士隐秉性恬淡，不以功名为念，每日只以观花种竹。酌酒吟诗为乐，倒是神仙一流人物。只是一件不足：年过半百，膝下无儿，只有一女，乳名英莲，年方三岁。
--------------------------------------------------------------------------------

【排名 2】相似度: 0.4320
Chunk 406 (长度: 175 字符)
内容: 又改题：“有凤来仪”赐名“潇湘馆”。“红香绿玉”改作“怡红快绿”，赐名“怡红院”。“蘅芷清芬”赐名“蘅芜院”。“杏帘在望”赐名“浣葛山庄”。正楼曰“大观楼”。东面飞楼曰“缀锦楼”。西面叙楼曰“含芳阁”。更有“蓼风轩”、“藕香榭”、“紫菱洲”、“荇叶渚”等名。匾额有“梨花春雨”、“桐剪秋风”、“荻芦夜雪”等名。又命旧有匾联不可摘去。于是先题一绝句云：
--------------------------------------------------------------------------------

【排名 3】相似度: 0.4250
Chunk 104 (长度: 459 字符)
内容: 原来这梨香院乃当日荣公暮年养静之所，小小巧巧，约有十馀间房舍，前厅后舍俱全。另有一门通街，薛蟠的家人就走此门出入：西南上又有一个角门，通着夹道子，出了夹道便是王夫人正房的东院了。每日或饭后或晚间，薛姨妈便过来，或与贾母闲谈，或与王夫人相叙。宝钗与黛玉、迎春妹妹等一处，或看书下棋，或做针黹，倒也十分相安。只是薛蟠起初原不欲在贾府中居住，生恐姨父管束，不得自在；无奈母亲执意在此，且贾宅中又十分殷勤苦留，只得

In [7]:
# 测试查询：蘅芜院是谁的住所？
from sentence_transformers import SentenceTransformer, util
import torch
import pickle

# 1. 加载模型
print("加载模型...")
model = SentenceTransformer('BAAI/bge-small-zh-v1.5')

# 2. 加载已生成的 embeddings 和 chunks
print("加载 embeddings 和 chunks...")
corpus_embeddings = torch.load('corpus_embeddings.pt')
with open('corpus_chunks.pkl', 'rb') as f:
    corpus = pickle.load(f)

print(f"已加载 {len(corpus)} 个chunks")
print(f"Embeddings 形状: {corpus_embeddings.shape}")

# 3. 测试查询
query = "蘅芜院是谁的住所？"
print(f"\n查询: {query}")

# 4. 生成查询的 embedding
query_embedding = model.encode(query, convert_to_tensor=True, normalize_embeddings=True)

# 5. 计算余弦相似度并排序
cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
top_results = torch.topk(cos_scores, k=10)  # 增加结果数量以便找到相关信息

# 6. 显示结果
print("\n" + "="*80)
print("Top 10 最相似的段落:")
print("="*80)
for i, (score, idx) in enumerate(zip(top_results[0], top_results[1]), 1):
    chunk_text = corpus[idx]
    # 检查是否包含关键词
    has_keywords = "蘅芜" in chunk_text or "宝钗" in chunk_text or "薛宝钗" in chunk_text
    marker = " ⭐" if has_keywords else ""
    
    print(f"\n【排名 {i}】相似度: {score:.4f}{marker}")
    print(f"Chunk {idx+1} (长度: {len(chunk_text)} 字符)")
    print(f"内容: {chunk_text[:400]}..." if len(chunk_text) > 400 else f"内容: {chunk_text}")
    print("-"*80)

加载模型...


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


加载 embeddings 和 chunks...
已加载 2636 个chunks
Embeddings 形状: torch.Size([2636, 512])

查询: 蘅芜院是谁的住所？

Top 10 最相似的段落:

【排名 1】相似度: 0.4413
Chunk 10 (长度: 209 字符)
内容: 按那石上书云：当日地陷东南，这东南有个姑苏城，城中阊门，最是红尘中一二等富贵风流之地。这阊门外有个十里街，街内有个仁清巷，巷内有个古庙，因地方狭窄，人皆呼作“葫芦庙”。庙旁住着一家乡宦，姓甄名费字士隐，嫡妻封氏，性情贤淑，深明礼义。家中虽不甚富贵，然本地也推他为望族了。因这甄士隐秉性恬淡，不以功名为念，每日只以观花种竹。酌酒吟诗为乐，倒是神仙一流人物。只是一件不足：年过半百，膝下无儿，只有一女，乳名英莲，年方三岁。
--------------------------------------------------------------------------------

【排名 2】相似度: 0.4320 ⭐
Chunk 406 (长度: 175 字符)
内容: 又改题：“有凤来仪”赐名“潇湘馆”。“红香绿玉”改作“怡红快绿”，赐名“怡红院”。“蘅芷清芬”赐名“蘅芜院”。“杏帘在望”赐名“浣葛山庄”。正楼曰“大观楼”。东面飞楼曰“缀锦楼”。西面叙楼曰“含芳阁”。更有“蓼风轩”、“藕香榭”、“紫菱洲”、“荇叶渚”等名。匾额有“梨花春雨”、“桐剪秋风”、“荻芦夜雪”等名。又命旧有匾联不可摘去。于是先题一绝句云：
--------------------------------------------------------------------------------

【排名 3】相似度: 0.4250 ⭐
Chunk 104 (长度: 459 字符)
内容: 原来这梨香院乃当日荣公暮年养静之所，小小巧巧，约有十馀间房舍，前厅后舍俱全。另有一门通街，薛蟠的家人就走此门出入：西南上又有一个角门，通着夹道子，出了夹道便是王夫人正房的东院了。每日或饭后或晚间，薛姨妈便过来，或与贾母闲谈，或与王夫人相叙。宝钗与黛玉、迎春妹妹等一处，或看书下棋，或做针黹，倒也十分相安。只是薛蟠起初原不欲在贾府中居住，生恐姨父管束，不得自在；无奈母亲执意在此，且贾宅中又十分殷勤

## LLM-as-Judge 检索质量评估

用 Qwen（通过 Fireworks AI）对每个检索结果打分，再汇总指标。

**流程：**
1. 读取 `Data/qa.jsonl` 中的43个测试问题
2. 对每个问题检索 Top-5 段落
3. 让 LLM 对每个段落打分（0-3分）
4. 计算 Precision@5、Hit Rate@5、MRR、NDCG@5

In [12]:
import json, torch, pickle
from sentence_transformers import SentenceTransformer, util

# 加载检索系统（已有 embeddings）
retrieval_model   = SentenceTransformer('BAAI/bge-small-zh-v1.5')
corpus_embeddings = torch.load('corpus_embeddings.pt', map_location='cpu')
with open('corpus_chunks.pkl', 'rb') as f:
    corpus = pickle.load(f)
print(f"✓ 语料库：{len(corpus)} 个 chunks")

# 加载测试集
qa_data = []
with open('Data/qa.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        qa_data.append(json.loads(line.strip()))
print(f"✓ 测试集：{len(qa_data)} 个问题")

def retrieve(question, top_k=5):
    q_emb  = retrieval_model.encode(question, convert_to_tensor=True, normalize_embeddings=True).cpu()
    scores = util.cos_sim(q_emb, corpus_embeddings)[0]
    top    = torch.topk(scores, k=top_k)
    return [
        {"rank": i+1, "chunk_id": int(idx),
         "retrieval_score": float(score), "text": corpus[int(idx)]}
        for i, (score, idx) in enumerate(zip(top.values, top.indices))
    ]

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ 语料库：2636 个 chunks
✓ 测试集：43 个问题


In [21]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("FIREWORKS_API_KEY"),
    base_url="https://api.fireworks.ai/inference/v1",
)

response = client.chat.completions.create(
    model="accounts/fireworks/models/deepseek-v3p2",
    messages=[
        {"role": "user", "content": """你是《红楼梦》信息检索评估专家。请评估下面的检索段落能否帮助回答问题。

问题：{蘅芜院是谁的住所?}
标准答案：{薛宝钗}

检索段落：
{赐名"蘅芜院"。宝钗住了蘅芜院，黛玉住了潇湘馆。}

评分标准（只看段落是否包含回答问题所需的信息）：
0 = 完全无关
1 = 轻微相关（背景信息，无法直接回答）
2 = 部分相关（包含有用线索）
3 = 高度相关（直接包含答案）

只返回 JSON，不要任何额外文字：{{"score": <0-3>, "reason": "<15字以内>"}}"""}
    ],
    extra_body={"reasoning_effort": "medium"},
)

msg = response.choices[0].message
# Reasoning content is returned in a separate field
reasoning = getattr(msg, "reasoning_content", None)
if reasoning is None and hasattr(msg, "model_extra"):
    reasoning = msg.model_extra.get("reasoning_content")

if reasoning:
    print("Reasoning:", reasoning)
print("Answer:", msg.content)


Reasoning: 首先，问题问的是：“蘅芜院是谁的住所？”标准答案是：“薛宝钗”。

检索段落是：“赐名"蘅芜院"。宝钗住了蘅芜院，黛玉住了潇湘馆。”

我需要评估这个检索段落能否帮助回答问题。评分标准是：

- 0 = 完全无关

- 1 = 轻微相关（背景信息，无法直接回答）

- 2 = 部分相关（包含有用线索）

- 3 = 高度相关（直接包含答案）

现在，分析检索段落：

- 段落说：“宝钗住了蘅芜院”。这直接提到蘅芜院的居住者是宝钗。

- 标准答案是薛宝钗，而段落中说“宝钗”，在《红楼梦》中，宝钗就是薛宝钗的简称。

所以，检索段落直接回答了问题：蘅芜院是宝钗的住所。

因此，这个段落直接包含答案。根据评分标准，3 = 高度相关（直接包含答案）。

评分标准强调“只看段落是否包含回答问题所需的信息”。这里，段落直接提供了答案，所以应该是3分。

现在，我需要返回JSON格式：{"score": <0-3>, "reason": "<15字以内>"}

- score: 应该是3。

- reason: 需要15字以内，中文。原因应该是段落直接回答了问题。例如，“直接提到宝钗住在蘅芜院”。

确保字数：中文字数。在15字以内。

可能的reason：“段落直接指出宝钗住蘅芜院”。检查字数：“段落直接指出宝钗住蘅芜院” 共9个字（段落、直接、指出、宝钗、住、蘅芜院），符合。

最后，只返回JSON，不要任何额外文字。
Answer: {"score": 3, "reason": "直接提到宝钗住在蘅芜院"}


In [ ]:
import os, json
from openai import OpenAI

# ── 评估参数配置 ─────────────────────────────────────────────
TOP_K       = 5
THRESHOLD   = 2
JUDGE_MODEL = "accounts/fireworks/models/deepseek-v3p2"

# ── Fireworks AI 客户端 ──────────────────────────────────────
fw_client = OpenAI(
    api_key=os.environ.get("FIREWORKS_API_KEY"),
    base_url="https://api.fireworks.ai/inference/v1",
)

# ── Judge Prompt ─────────────────────────────────────────────
JUDGE_PROMPT = """你是《红楼梦》信息检索评估专家。请评估下面的检索段落能否帮助回答问题。

问题：{question}
标准答案：{answer}

检索段落：
{chunk}

评分标准（只看段落是否包含回答问题所需的信息）：
0 = 完全无关
1 = 轻微相关（背景信息，无法直接回答）
2 = 部分相关（包含有用线索）
3 = 高度相关（直接包含答案）

只返回 JSON，不要任何额外文字：{{"score": <0-3>, "reason": "<15字以内>"}}"""

def judge_chunk(question, answer, chunk_text):
    prompt = JUDGE_PROMPT.format(question=question, answer=answer, chunk=chunk_text)
    try:
        resp = fw_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": prompt}]
        )
        msg = resp.choices[0].message

        reasoning = getattr(msg, "reasoning_content", None)
        if reasoning is None and hasattr(msg, "model_extra"):
            reasoning = msg.model_extra.get("reasoning_content")

        result = json.loads(msg.content)
        return int(result["score"]), result["reason"], reasoning
    except Exception as e:
        return 0, f"错误:{str(e)[:30]}", None

# 冒烟测试
score, reason, reasoning = judge_chunk(
    "蘅芜院是谁的住所？", "薛宝钗",
    '赐名"蘅芜院"。宝钗住了蘅芜院，黛玉住了潇湘馆。'
)
# if reasoning:
#     print("Reasoning:", reasoning)
print(f"冒烟测试 → 得分: {score}/3，理由: {reason}")

Reasoning: 首先，问题是：“蘅芜院是谁的住所？”标准答案是：薛宝钗。

检索段落是：“赐名'蘅芜院'。宝钗住了蘅芜院，黛玉住了潇湘馆。”

我需要根据评分标准评估这个检索段落能否帮助回答问题。评分标准是：

- 0 = 完全无关

- 1 = 轻微相关（背景信息，无法直接回答）

- 2 = 部分相关（包含有用线索）

- 3 = 高度相关（直接包含答案）

段落明确说：“宝钗住了蘅芜院”。这直接回答了问题：蘅芜院是薛宝钗的住所。

所以，段落直接包含了答案。因此，评分应该是3 = 高度相关（直接包含答案）。

现在，输出格式必须是JSON：{"score": <0-3>, "reason": "<15字以内>"}

- score：根据评估，应该是3。

- reason：需要提供一个15字以内的简短理由。既然段落直接包含答案，理由可以是“段落直接指明宝钗住蘅芜院”。

确保理由在15字以内：中文字符计数。“段落直接指明宝钗住蘅芜院” – 让我们数一下：段落(2字)、直接(2字)、指明(2字)、宝钗(2字)、住(1字)、蘅芜院(3字)。总字数是2+2+2+2+1+3=12字，在15字以内。或者更简洁：“直接说明宝钗住所”。

但为了准确，使用与评估相关的理由。既然直接包含答案，就说“直接包含答案”。

最终JSON：{"score": 3, "reason": "直接包含答案"}

检查是否符合要求：只返回JSON，不要任何额外文字。所以，输出应该是纯JSON。
冒烟测试 → 得分: 3/3，理由: 直接包含答案
